In [15]:
import numpy as np
import matplotlib.pyplot
import gymnasium as gym
import pandas as pd
import time
from numpy.testing import verbose
from stable_baselines3 import SAC, TD3
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.results_plotter import ts2xy, load_results

In [19]:
env_id = 'LunarLanderContinuous-v3'
seed = 42
total_time = 500_000
n_evals = 20
np.random.seed(seed)

In [20]:
def train_and_eval(algo, env_id, log_dir, n_steps = total_time, seed =seed, n_evals = n_evals):
    t0 = time.time()
    train_env = make_vec_env(env_id, n_envs=8, seed=42)
    eval_env = make_vec_env(env_id, n_envs=1, seed = seed*8)
    model = algo('MlpPolicy', train_env, verbose=0, seed=seed)
    model.learn(total_timesteps = n_steps)
    mean_reward, std_dev = evaluate_policy(model, eval_env, n_eval_episodes=n_steps, deterministic=True)
    train_env.close()
    eval_env.close()
    return mean_reward, std_dev, t0

In [ ]:
sac_model, sac_mean_reward, sac_std_dev, t0 = train_and_eval(SAC, env_id, './logs/SAC')
tf = time.time()
sac_model.save('SAC LunarLander model')
print(f'The reward after running SAC over this environment is {sac_mean_reward} +- {sac_std_dev} and it took {tf - t0} secs')

In [ ]:
def train_and_eval(algo, env_id, time_steps, log_dir, n_evals = n_evals, seed=seed):
    t0 = time.time()
    train_env = make_vec_env(env_id, n_envs=8, seed = seed)
    eval_env = make_vec_env(env_id, n_envs=1, seed =seed)
    model = TD3('MlpPolicy', train_env, learning_rate = 0.01, batch_size = 256, gradient_steps = 16, verbose=0, seed=seed)
    model.learn(total_timesteps = time_steps)
    mean_reward, std_dev = evaluate_policy(model, eval_env, n_eval_episodes = time_steps, deterministic=True)
    train_env.close()
    eval_env.close()
    return mean_reward, std_dev

time_steps = 500_000

TD3_model, TD3_mean_reward, TD3_std_dev, TD3_time = train_and_eval(TD3, env_id, time_steps, './logs/TD3')
TD3_model.save('TD3 LunarLander')
tf = time.time()
print(f"TD3 model is successfully saved and run! with rewards = {TD3_mean_reward}+-{TD3_std_dev} in {tf-TD3_time} secs!")